# Phase 2 — Demo A: within-dataset temporal leakage (2018)


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


Random vs chronological vs leave-one-day-out on CSE-CIC-IDS2018. The gap between random and time-aware splits is the headline result.

In [ ]:
import pandas as pd, numpy as np
from driftbench import (config, make_model, compute_metrics, metrics_frame,
                        random_split, chronological_split, leave_one_day_out)
ds = 'CSE-CIC-IDS2018'
p = config.INTERIM_DIR / ds
X = pd.read_parquet(p / 'features.parquet')
y = pd.read_parquet(p / 'labels.parquet')['label']
meta = pd.read_parquet(p / 'metadata.parquet')
MODEL = 'rf'  # repeat for 'lgbm' and 'mlp'


In [ ]:
def fit_eval(tr, te):
    mdl = make_model(MODEL, config.SEED_ANCHOR).fit(X.iloc[tr], y.iloc[tr])
    proba = mdl.predict_proba(X.iloc[te]); pred = mdl.predict(X.iloc[te])
    return compute_metrics(y.iloc[te], pred, proba, mdl.classes_), pred

rows = {}
rs = random_split(len(X), y, seed=config.SEED_ANCHOR);  rows['random'],   pred_rand  = fit_eval(rs['train'], rs['test'])
cs = chronological_split(meta);                         rows['chrono'],   pred_chron = fit_eval(cs['train'], cs['test'])
# leave-one-day-out: average over folds
import numpy as np
lodo = leave_one_day_out(meta)
fold_metrics = [fit_eval(f['train'], f['test'])[0] for _, f in lodo]
rows['lodo'] = {k: float(np.nanmean([fm[k] for fm in fold_metrics])) for k in fold_metrics[0]}

tbl = metrics_frame(rows)
tbl.to_csv(config.RESULTS_DIR / f'demoA_{ds}_{MODEL}.csv')
tbl[['macro_f1','mcc','ece']]
